# PyTorch: Regreesion using FastAI

Dataset: BIWI_HEAD_POSE
(predicting the center of a human face)

In [ ]:
from fastai.vision.all import *
import numpy as np

# Prepare Dataset

In [ ]:
path = untar_data(URLs.BIWI_HEAD_POSE)

In [ ]:
cal = np.genfromtxt(path/"01"/"rgb.cal", skip_footer=6)

def img2pose(x): return Path(f'{str(x)[:-7]}pose.txt')

def get_ctr(f):
    ctr = np.genfromtxt(img2pose(f), skip_header=3)
    c1 = ctr[0] * cal[0][0]/ctr[2] + cal[0][2]
    c2 = ctr[1] * cal[1][1]/ctr[2] + cal[1][2]
    return tensor([c1,c2])

dls = DataBlock(
    blocks=(ImageBlock, PointBlock),
    get_items=get_image_files,
    get_y=get_ctr,
    splitter=FuncSplitter(lambda o: o.parent.name=='13'),
    batch_tfms=aug_transforms(size=(240,320)), 
).dataloaders(path)

In [ ]:
dls.show_batch(max_n=9, figsize=(8,6))

# Train Model

In [ ]:
learn = vision_learner(dls, resnet18, y_range=(-1,1))

In [ ]:
learn.fine_tune(3, base_lr=1e-2)

# Evaluate Model

In [ ]:
learn.show_results(ds_idx=1, nrows=3, figsize=(6,8))